# SOLARNESIA: Spatial Model & GIS Analytics for National Solar Power Plant (PLTS) Distribution
**Mata Kuliah:** IMT01303501 - Komputer Grafis (3 SKS)  
**Kelompok 6:** Wesley Goeinawan (0706012410011), Stevanus Ivan Santoso (0706012410029), Nicholas Gerwin Mawardji (0706012410015)  
**Program Studi:** Informatika, Universitas Ciputra Surabaya (2026/2027)

---
### Ringkasan Alur Pemrosesan Spasial
1. **Layer 1 (2D MultiPolygons):** Batas daratan kepulauan dan batas administrasi kabupaten/provinsi (Choropleth rata-rata potensi PVOUT nasional).
2. **Layer 2 (0D Points & Buffers):** Titik observasi Global Solar Atlas darat murni (3.802 titik) setelah *Spatial Clip* mengeliminasi 1.160 titik laut.
3. **5 km Hotspot Clustering:** Pengelompokan titik kandidat berdekatan (≤ 5 km) menjadi sentra hotspot terintegrasi (DBSCAN).
4. **Operasi 1 (Buffer Metrik):** Proyeksi EPSG:3857 dengan koreksi $\cos(\text{lat})$, pembentukan *footprint* 3 km `buffer(3000)` dan koridor transmisi 25 km `buffer(25000)` (standar PLN 20 kV).
5. **Operasi 2 (Spatial Join & Overlay):** Menghitung keterjangkauan 3.289 permukiman darat serta mengekstrak daftar kecamatan/kabupaten yang tercakup secara interaktif.


In [17]:
# Cell 0: Setup & Library Imports
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch, Circle
from matplotlib.lines import Line2D
from shapely.geometry import Point, Polygon
from sklearn.cluster import DBSCAN
import folium

print(f"GeoPandas: {gpd.__version__} | Pandas: {pd.__version__} | Folium: {folium.__version__}")
os.makedirs("outputs", exist_ok=True)
print("Output directory ready.")


GeoPandas: 1.2.0 | Pandas: 3.0.6 | Folium: 0.20.0
Output directory ready.


In [18]:
# Cell 1: Ingestion & GeoDataFrame Construction
DATA_DIR = "data"

# 1. Load Solar Observation Points (Global Solar Atlas)
df_raw = pd.read_csv(os.path.join(DATA_DIR, "indonesia_solar_data.csv"))
raw_pts = gpd.GeoDataFrame(
    df_raw,
    geometry=gpd.points_from_xy(df_raw['longitude'], df_raw['latitude']),
    crs="EPSG:4326"
)

# 2. Load Administrative Boundaries (Provinces & Kabupaten)
gdf_prov = gpd.read_file(os.path.join(DATA_DIR, "indonesia_provinces.geojson"))
gdf_kab = gpd.read_file(os.path.join(DATA_DIR, "indonesia_kabupaten.geojson"))

print(f"Raw Solar Points loaded: {len(raw_pts)} points | CRS: {raw_pts.crs}")
print(f"Provinces loaded: {len(gdf_prov)} | Kabupaten loaded: {len(gdf_kab)}")

assert len(raw_pts) == 4962, f"Expected 4962 points, got {len(raw_pts)}"
assert raw_pts.crs.to_epsg() == 4326
print("Input verification passed.")


Raw Solar Points loaded: 4962 points | CRS: EPSG:4326
Provinces loaded: 34 | Kabupaten loaded: 519
Input verification passed.


In [19]:
# Cell 2: Spatial Clip / Land-Masking (Op 0)
# Mengeliminasi titik-titik observasi yang jatuh di perairan laut menggunakan poligon daratan kabupaten
land_sj = gpd.sjoin(raw_pts, gdf_kab[['shapeName', 'geometry']], how='inner', predicate='intersects')
land_pts = raw_pts.loc[land_sj.index.unique()].copy()

ocean_pts_count = len(raw_pts) - len(land_pts)
print(f"Total Raw Points       : {len(raw_pts)}")
print(f"Eliminated Ocean Points: {ocean_pts_count} (Dropped via Land-Masking)")
print(f"Pure Land Points       : {len(land_pts)}")

assert len(land_pts) == 3802, f"Expected 3802 land points, got {len(land_pts)}"
assert ocean_pts_count == 1160, f"Expected 1160 ocean points, got {ocean_pts_count}"
print("Spatial Clip verification passed.")


Total Raw Points       : 4962
Eliminated Ocean Points: 1160 (Dropped via Land-Masking)
Pure Land Points       : 3802
Spatial Clip verification passed.


In [20]:
# Cell 3: Site Selection & Role Classification (Layer 2)
# Ambang batas: PVOUT >= Persentil ke-75 (1.430 kWh/kWp) dan Suhu Rata-rata <= 27,5 °C
p75_pvout = 1430.0
max_temp = 27.5

is_plts = (land_pts['pvout'] >= p75_pvout) & (land_pts['temp'] <= max_temp)
plts_candidates = land_pts[is_plts].copy()
demand_points = land_pts[~is_plts].copy()

plts_candidates['role'] = 'PLTS_CANDIDATE'
demand_points['role'] = 'DEMAND'

print(f"PLTS Candidate Points: {len(plts_candidates)} (PVOUT >= {p75_pvout} & Temp <= {max_temp}°C)")
print(f"Demand Points (Darat): {len(demand_points)}")

assert len(plts_candidates) == 513, f"Expected 513 PLTS candidates, got {len(plts_candidates)}"
assert len(demand_points) == 3289, f"Expected 3289 demand points, got {len(demand_points)}"
print("Role classification verified.")


PLTS Candidate Points: 513 (PVOUT >= 1430.0 & Temp <= 27.5°C)
Demand Points (Darat): 3289
Role classification verified.


In [21]:
# Cell 4: 5 km Proximity Hotspot Clustering (DBSCAN)
# Mengelompokkan titik kandidat yang berjarak <= 5 km menjadi satu klaster sentra energi terpadu
plts_3857 = plts_candidates.to_crs("EPSG:3857")
coords_m = np.column_stack([plts_3857.geometry.x, plts_3857.geometry.y])

# Clustering dengan radius ketetanggaan 5.000 meter
db = DBSCAN(eps=5000, min_samples=1, metric='euclidean').fit(coords_m)
plts_candidates['hotspot_id'] = [f"HOTSPOT-{cid:03d}" for cid in db.labels_]
plts_3857['hotspot_id'] = plts_candidates['hotspot_id']

unique_hotspots = plts_candidates['hotspot_id'].nunique()
cluster_sizes = plts_candidates['hotspot_id'].value_counts()
multi_site_hotspots = (cluster_sizes > 1).sum()

print(f"Total Kandidat PLTS      : {len(plts_candidates)} titik")
print(f"Total Hotspot Sentra     : {unique_hotspots} klaster")
print(f"Hotspot Multi-titik (<=5km): {multi_site_hotspots} klaster terpadu")

assert unique_hotspots == 480, f"Expected 480 hotspots, got {unique_hotspots}"
assert multi_site_hotspots == 31, f"Expected 31 multi-site clusters, got {multi_site_hotspots}"
print("5km Proximity Clustering verified.")


Total Kandidat PLTS      : 513 titik
Total Hotspot Sentra     : 480 klaster
Hotspot Multi-titik (<=5km): 31 klaster terpadu
5km Proximity Clustering verified.


In [22]:
# Cell 5: Transformasi CRS & Pembuatan Buffer Metrik (Op 1)
# Transformasi ke EPSG:3857 dengan koreksi distorsi Mercator: R_adj = R / cos(latitude)
demand_3857 = demand_points.to_crs("EPSG:3857")

lats_plts = plts_candidates['latitude'].values
rad_adj_3k = 3000.0 / np.cos(np.radians(lats_plts))
rad_adj_25k = 25000.0 / np.cos(np.radians(lats_plts))

# Buffer 3 km (Footprint Lahan) dan Buffer 25 km (Standar Distribusi 20 kV PLN)
plts_3857['footprint_3km'] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_3k)]
plts_3857['buffer_25km'] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_25k)]

# Buffer Sensitivitas 10 km dan 40 km
rad_adj_10k = 10000.0 / np.cos(np.radians(lats_plts))
rad_adj_40k = 40000.0 / np.cos(np.radians(lats_plts))
plts_3857['buffer_10km'] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_10k)]
plts_3857['buffer_40km'] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_40k)]

# Poligon Union Koridor Distribusi
union_25km = gpd.GeoSeries(plts_3857['buffer_25km'], crs="EPSG:3857").union_all()
union_10km = gpd.GeoSeries(plts_3857['buffer_10km'], crs="EPSG:3857").union_all()
union_40km = gpd.GeoSeries(plts_3857['buffer_40km'], crs="EPSG:3857").union_all()

print("Buffer polygons generated with latitude-scaling compensation.")


Buffer polygons generated with latitude-scaling compensation.


In [23]:
# Cell 6: Analisis Keterjangkauan Permukiman Spasial (Op 2)
# Titik demand diuji posisinya terhadap poligon buffer distribusi menggunakan predikat intersects / within
demand_3857['covered_25km'] = demand_3857.geometry.within(union_25km)
demand_3857['covered_10km'] = demand_3857.geometry.within(union_10km)
demand_3857['covered_40km'] = demand_3857.geometry.within(union_40km)

reached_25 = int(demand_3857['covered_25km'].sum())
unreached_25 = len(demand_3857) - reached_25
pct_25 = (reached_25 / len(demand_3857)) * 100

reached_10 = int(demand_3857['covered_10km'].sum())
pct_10 = (reached_10 / len(demand_3857)) * 100

reached_40 = int(demand_3857['covered_40km'].sum())
pct_40 = (reached_40 / len(demand_3857)) * 100

print(f"Jangkauan 10 km (Sensitivitas)   : {reached_10} titik ({pct_10:.1f}%)")
print(f"Jangkauan 25 km (Standar PLN 20kV): {reached_25} titik ({pct_25:.1f}%)")
print(f"Jangkauan 40 km (Sensitivitas)   : {reached_40} titik ({pct_40:.1f}%)")
print(f"Permukiman Belum Terjangkau (>25km): {unreached_25} titik ({(unreached_25/len(demand_3857))*100:.1f}%)")

assert reached_25 == 325, f"Expected 325, got {reached_25}"
assert unreached_25 == 2964, f"Expected 2964, got {unreached_25}"
assert round(pct_25, 1) == 9.9
assert reached_10 == 49
assert reached_40 == 605
print("Demand reach statistics 100% match proposal.")


Jangkauan 10 km (Sensitivitas)   : 49 titik (1.5%)
Jangkauan 25 km (Standar PLN 20kV): 325 titik (9.9%)
Jangkauan 40 km (Sensitivitas)   : 605 titik (18.4%)
Permukiman Belum Terjangkau (>25km): 2964 titik (90.1%)
Demand reach statistics 100% match proposal.


In [24]:
# Cell 7: Ekstraksi Cakupan Wilayah Administratif per Titik / Hotspot
# Mengiriskan buffer 25 km tiap kandidat dengan poligon batas kabupaten/kota
kab_3857 = gdf_kab.to_crs("EPSG:3857")

covered_areas_list = []
print("Pre-calculating 25km administrative reach for each hotspot...")

for idx, row in plts_3857.iterrows():
    buf_geom = row['buffer_25km']
    intersected_kabs = kab_3857[kab_3857.geometry.intersects(buf_geom)]['shapeName'].unique()
    covered_str = ", ".join(sorted(intersected_kabs))
    covered_areas_list.append(covered_str)

plts_candidates['covered_areas_25km'] = covered_areas_list
plts_3857['covered_areas_25km'] = covered_areas_list

# Cakupan Nasional
all_covered_kabs = kab_3857[kab_3857.geometry.intersects(union_25km)]['shapeName'].unique()
print(f"Total Kabupaten/Kota Terjangkau Cincin PLTS: {len(all_covered_kabs)} dari {len(gdf_kab)} nasional")

# Uji Cek Titik Bali Selatan (Denpasar & Badung)
bali_pt = plts_candidates[(plts_candidates['latitude'] > -8.8) & (plts_candidates['latitude'] < -8.5) & (plts_candidates['longitude'] > 115.0) & (plts_candidates['longitude'] < 115.3)].iloc[0]
print("\n[Contoh Klik Titik Bali Selatan] ({:.3f}° BT, {:.3f}° LS):".format(bali_pt.longitude, bali_pt.latitude))
print(f"  Hotspot ID     : {bali_pt.hotspot_id}")
print(f"  Potensi Radiasi: {bali_pt.pvout:.1f} kWh/kWp | Suhu: {bali_pt.temp:.1f}°C")
print(f"  Wilayah 25 km  : {bali_pt.covered_areas_25km}")

assert len(all_covered_kabs) == 294, f"Expected 294 covered kabupaten, got {len(all_covered_kabs)}"
assert "Kota Denpasar" in bali_pt.covered_areas_25km and "Badung" in bali_pt.covered_areas_25km
print("\nAdministrative coverage assertions passed.")


Pre-calculating 25km administrative reach for each hotspot...
Total Kabupaten/Kota Terjangkau Cincin PLTS: 294 dari 519 nasional

[Contoh Klik Titik Bali Selatan] (115.179° BT, -8.654° LS):
  Hotspot ID     : HOTSPOT-393
  Potensi Radiasi: 1557.4 kWh/kWp | Suhu: 26.4°C
  Wilayah 25 km  : Badung, Bangli, Gianyar, Klungkung, Kota Denpasar, Tabanan

Administrative coverage assertions passed.


In [25]:
# Cell 8: Rangkuman Metrik & Ekspor Tabel 1
summary_table1 = pd.DataFrame([
    {
        'Radius Buffer': '10 km (Sensitivitas)',
        'Permukiman Terjangkau': reached_10,
        'Persentase': f"{pct_10:.1f}%".replace('.', ',')
    },
    {
        'Radius Buffer': '25 km (Standar Resmi PLN)',
        'Permukiman Terjangkau': reached_25,
        'Persentase': f"{pct_25:.1f}%".replace('.', ',')
    },
    {
        'Radius Buffer': '40 km (Sensitivitas)',
        'Permukiman Terjangkau': reached_40,
        'Persentase': f"{pct_40:.1f}%".replace('.', ',')
    }
])

summary_table1.to_csv("outputs/solarnesia_table1_summary.csv", index=False)
pd.Series(sorted(all_covered_kabs), name="Kabupaten_Kota_Terlayani").to_csv("outputs/solarnesia_covered_kabupaten.csv", index=False)

print("Tabel 1: Hasil Analisis Spatial Join Jangkauan Layanan PLTS:")
print(summary_table1.to_string(index=False))
print("\nFiles saved: outputs/solarnesia_table1_summary.csv & outputs/solarnesia_covered_kabupaten.csv")


Tabel 1: Hasil Analisis Spatial Join Jangkauan Layanan PLTS:
            Radius Buffer  Permukiman Terjangkau Persentase
     10 km (Sensitivitas)                     49       1,5%
25 km (Standar Resmi PLN)                    325       9,9%
     40 km (Sensitivitas)                    605      18,4%

Files saved: outputs/solarnesia_table1_summary.csv & outputs/solarnesia_covered_kabupaten.csv


In [26]:
# Cell 9: Visualisasi Komputer Grafis 2D Statis (Matplotlib)
# Menggabungkan Layer 1 (Choropleth Vektor) dan Layer 2 (Titik Diskrit & Koridor Buffer)
fig = plt.figure(figsize=(16, 9.5), facecolor='#090d16')
ax = fig.add_axes([0.05, 0.18, 0.90, 0.74], facecolor='#090d16')

ax.set_xlim(94.5, 141.5)
ax.set_ylim(-11.5, 6.5)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', color='#1e293b', alpha=0.5, linewidth=0.7)
ax.tick_params(colors='#64748b', labelsize=9)
for spine in ax.spines.values():
    spine.set_color('#1e293b')

# 1. Layer 1: Provincial Base Map & Choropleth
prov_means = gpd.sjoin(land_pts, gdf_prov, how='inner', predicate='intersects').groupby('state')['pvout'].mean()
gdf_prov['mean_pvout'] = gdf_prov['state'].map(prov_means).fillna(land_pts['pvout'].mean())

cmap = plt.cm.YlOrRd
norm = mcolors.Normalize(vmin=1250, vmax=1620)
gdf_prov.plot(column='mean_pvout', cmap=cmap, norm=norm, edgecolor='#ffffff', linewidth=0.6, ax=ax, alpha=0.85)

# 2. Layer 2: 25 km Buffer Zones (Translucent Cyan Corridors)
gpd.GeoDataFrame(geometry=plts_3857['buffer_25km'], crs="EPSG:3857").to_crs("EPSG:4326").plot(
    ax=ax, facecolor='#38bdf8', edgecolor='#0284c7', alpha=0.22, linewidth=0.8
)

# 3. Layer 2: Demand Points (Unreached Slate vs Reached Emerald)
demand_4326 = demand_3857.to_crs("EPSG:4326")
demand_4326[~demand_4326['covered_25km']].plot(ax=ax, color='#64748b', markersize=14, alpha=0.55, label='Demand Belum Terjangkau (2.964)')
demand_4326[demand_4326['covered_25km']].plot(ax=ax, color='#10b981', markersize=26, alpha=0.85, edgecolors='#ffffff', linewidth=0.5, label='Demand Terjangkau 25 km (325)')

# 4. Layer 2: PLTS Candidate Sites (Glowing Amber Stars)
plts_candidates.plot(ax=ax, color='#fbbf24', marker='*', markersize=90, edgecolors='#ffffff', linewidth=0.6, label='Kandidat PLTS (513 / 480 Hotspot)')

# Inset Zoom: Jawa Timur, Madura & Bali Detail
ax_ins = fig.add_axes([0.65, 0.55, 0.28, 0.32], facecolor='#0d1527')
ax_ins.set_title("Detail Jangkauan: Jawa Timur, Madura & Bali", color='#f8fafc', fontsize=9.5, fontweight='bold', pad=6)
for s in ax_ins.spines.values():
    s.set_color('#38bdf8')
    s.set_linewidth(1.2)

sub_prov = gdf_prov[gdf_prov['state'].isin(['Jawa Timur', 'Bali', 'Nusa Tenggara Barat'])]
sub_prov.plot(ax=ax_ins, column='mean_pvout', cmap=cmap, norm=norm, edgecolor='#ffffff', linewidth=0.7)

gpd.GeoDataFrame(geometry=plts_3857['buffer_25km'], crs="EPSG:3857").to_crs("EPSG:4326").plot(
    ax=ax_ins, facecolor='#38bdf8', edgecolor='#38bdf8', alpha=0.28, linewidth=1.0
)
demand_4326[demand_4326['covered_25km']].plot(ax=ax_ins, color='#10b981', markersize=35, edgecolors='#ffffff', linewidth=0.6)
demand_4326[~demand_4326['covered_25km']].plot(ax=ax_ins, color='#64748b', markersize=16, alpha=0.6)
plts_candidates.plot(ax=ax_ins, color='#fbbf24', marker='*', markersize=110, edgecolors='#ffffff', linewidth=0.8)

ax_ins.set_xlim(111.0, 116.0)
ax_ins.set_ylim(-9.0, -6.8)
ax_ins.tick_params(colors='#64748b', labelsize=6.5)

# Colorbar & Legend
cbar_ax = fig.add_axes([0.06, 0.08, 0.30, 0.022])
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, orientation='horizontal')
cb.set_label('Rata-rata Potensi Radiasi Surya (PVOUT - kWh/kWp/tahun)', color='#e2e8f0', fontsize=8)
cb.ax.tick_params(labelsize=7.5, colors='#94a3b8')

leg = ax.legend(loc='lower right', facecolor='#0f172a', edgecolor='#334155', fontsize=8.5)
for t in leg.get_texts():
    t.set_color('#f1f5f9')

fig.text(0.05, 0.94, "SOLARNESIA: MODEL INTEGRASI SPASIAL DUA LAPIS PLTS NASIONAL", fontsize=16, fontweight='bold', color='#f8fafc')
fig.text(0.05, 0.91, "Layer 1: Choropleth Poligon Daratan | Layer 2: 513 Titik PLTS (480 Hotspot) & Koridor Feeder 25 km PLN 20 kV", fontsize=10, color='#38bdf8')

out_png = "outputs/solarnesia_coverage_map.png"
fig.savefig(out_png, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f"Composite 2D visualization exported to: {out_png}")


Composite 2D visualization exported to: outputs/solarnesia_coverage_map.png


In [27]:
# Cell 10: Peta Spasial Interaktif & Inspeksi Klik (Folium)
# Klik pada sembarang titik (misal Bali Selatan) untuk melihat nama-nama kecamatan/kabupaten dalam radius 25 km
print("Generating interactive Folium web map...")
# Center map on Indonesia (Bali & Java focal area)
center_lat, center_lon = -8.40, 115.20
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles="OpenStreetMap",
    control_scale=True
)
# 1. Add 25 km Buffer Polygons (Translucent Cyan)
buffer_layer = folium.FeatureGroup(name="Koridor Distribusi 25 km (PLN 20 kV)")
for idx, row in plts_candidates.iterrows():
    buf_circle = folium.Circle(
        location=[row['latitude'], row['longitude']],
        radius=25000,
        color="#38bdf8",
        weight=1.5,
        fill=True,
        fill_color="#38bdf8",
        fill_opacity=0.15,
        tooltip=f"<b>Hotspot:</b> {row['hotspot_id']}<br><b>Radius:</b> 25 km"
    )
    buf_circle.add_to(buffer_layer)
buffer_layer.add_to(m)
# 2. Add Reached Demand Points (Emerald Dots)
demand_layer = folium.FeatureGroup(name="Permukiman Terjangkau 25 km (325 Titik)")
for idx, row in demand_4326[demand_4326['covered_25km']].iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3.5,
        color="#10b981",
        fill=True,
        fill_color="#10b981",
        fill_opacity=0.85,
        tooltip=f"Demand ID: {idx} | Terjangkau"
    ).add_to(demand_layer)
demand_layer.add_to(m)
# 3. Add PLTS Candidate Markers with Click Popups
plts_layer = folium.FeatureGroup(name="Kandidat PLTS & Hotspot Sentra (513 Titik)")
for idx, row in plts_candidates.iterrows():
    popup_html = f"""
    <div style='font-family: Arial, sans-serif; font-size: 12px; width: 260px; line-height: 1.5;'>
        <b style='color: #d97706; font-size: 13px;'>☀️ {row['hotspot_id']}</b><br>
        <b>Koordinat:</b> {row['latitude']:.4f}° LS, {row['longitude']:.4f}° BT<br>
        <b>Potensi PVOUT:</b> <span style='color: #059669; font-weight: bold;'>{row['pvout']:.1f} kWh/kWp</span><br>
        <b>Suhu Rata-rata:</b> {row['temp']:.1f} °C<br>
        <hr style='margin: 6px 0; border: 0; border-top: 1px solid #e5e7eb;'>
        <b>Wilayah Terjangkau (Radius 25 km):</b><br>
        <span style='color: #1e3a8a; font-size: 11px;'>{row['covered_areas_25km']}</span>
    </div>
    """
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color="#ffffff",
        weight=1.5,
        fill=True,
        fill_color="#f59e0b",
        fill_opacity=0.95,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"<b>Klik untuk Inspeksi:</b> {row['hotspot_id']} (PVOUT: {row['pvout']:.0f})"
    ).add_to(plts_layer)
plts_layer.add_to(m)
folium.LayerControl().add_to(m)
out_html = "outputs/solarnesia_interactive_map.html"
m.save(out_html)
print(f"Interactive web map saved to: {out_html}")
print("Click on any amber PLTS marker (e.g. Southern Bali) to inspect the 25km covered areas.")
m


Generating interactive Folium web map...
Interactive web map saved to: outputs/solarnesia_interactive_map.html
Click on any amber PLTS marker (e.g. Southern Bali) to inspect the 25km covered areas.
